In [3]:
!python --version

Python 3.12.5


In [ ]:
import yfinance as yf
from stock_prediction import logger
import pandas as pd
from stock_prediction.entity.config_entity import DataIngestionConfig
from stock_prediction.utils.common import save_csv

class DataIngestion:
    def __init__(self, config: DataIngestionConfig):
        self.config =  config
    
    def fetch_file(self):
        try:
            
            logger.info(f"Data downloading for ticker: {self.config.ticker} from \
                {self.config.start_date} to {self.config.end_date}")
            data = yf.download(
                tickers=self.config.ticker,
                start=self.config.start_date,
                end=self.config.end_date
            )
            logger.info(f"Data downloaded successfully for ticker: {self.config.ticker} from \
                {self.config.start_date} to {self.config.end_date}")
            if not (data.empty):
                data = self._preprocesss(data)          
                save_csv(data, self.config.raw_data_file)

            else:
                logger.error(f"No data found for ticker: {self.config.ticker} from \
                    {self.config.start_date} to {self.config.end_date}") 
                raise ValueError(f"No data found for ticker: {self.config.ticker} from \
                    {self.config.start_date} to {self.config.end_date}")
        except Exception as e:
            logger.error(f"Error while downloading data for ticker: {self.config.ticker} ")
            raise e

        
        
    def _preprocesss(self, data: pd.DataFrame) -> pd.DataFrame:
        if isinstance(data.columns, pd.MultiIndex):
            data.columns = data.columns.droplevel(1)
        data = data.reset_index()
        # data = pd.read_csv(data, parse_dates="Date")
        data.columns = data.columns.str.lower() 
        data.to_datetime("date")
        return data

In [44]:
import pandas as pd
import yfinance as yf
data = yf.download("AAPL", start="2001-01-01", end="2026-08-04");
data.columns = data.columns.droplevel(1);
data = data.reset_index()
data.columns = data.columns.str.lower() 
data
# data.reset_index()
# data.head()
# data.info()
data = data.set_index("date")
data.info()


[*********************100%***********************]  1 of 1 completed

<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 6433 entries, 2001-01-02 to 2026-08-03
Data columns (total 5 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   close   6433 non-null   float64
 1   high    6433 non-null   float64
 2   low     6433 non-null   float64
 3   open    6433 non-null   float64
 4   volume  6433 non-null   int64  
dtypes: float64(4), int64(1)
memory usage: 301.5 KB


Price,Close,High,Low,Open,Volume
Date,,,,,
2001-01-02,0.222453,0.228061,0.217780,0.222453,452312000
2001-01-03,0.244885,0.249558,0.215911,0.216845,817073600
2001-01-04,0.255167,0.276664,0.251428,0.271290,739396000
2001-01-05,0.244885,0.259840,0.240211,0.253297,412356000
2001-01-08,0.247689,0.253998,0.238342,0.253297,373699200
...,...,...,...,...,...
2026-07-28,339.786926,342.594533,335.310806,339.736982,51859000
2026-07-29,337.898590,344.273097,337.059318,339.437272,56090800
2026-07-30,333.142670,334.461540,329.305982,332.812967,74817800


In [10]:
yf.download?

Signature:
yf.download(
    tickers,
    start=None,
    end=None,
    actions=False,
    threads=True,
    ignore_tz=None,
    group_by='column',
    auto_adjust=True,
    back_adjust=False,
    repair=False,
    keepna=False,
    progress=True,
    period='1mo if start & end None',
    interval='1d',
    prepost=False,
    rounding=False,
    timeout=10,
    session=None,
    multi_level_index=True,
) -> Optional[pandas.core.frame.DataFrame]
Docstring:
Download yahoo tickers
:Parameters:
    tickers : str, list
        List of tickers to download
    period : str
        Valid periods: 1d,5d,1mo,3mo,6mo,1y,2y,5y,10y,ytd,max
        Default: '1mo' if start & end None
        Either Use period parameter or use start and end
    interval : str
        Valid intervals: 1m,2m,5m,15m,30m,60m,90m,1h,1d,5d,1wk,1mo,3mo
        Intraday data cannot extend last 60 days
    start: str
        Download start date string (YYYY-MM-DD) or _datetime, inclusive.
        Default is 99 years ago
       

In [9]:
df = data.reset_index(drop="index")
df.head()

Price,Date,Close,High,Low,Open,Volume
Ticker,,AAPL,AAPL,AAPL,AAPL,AAPL
0,2001-01-02,0.222645,0.228257,0.217968,0.222645,452312000
1,2001-01-03,0.245097,0.249774,0.216097,0.217032,817073600
2,2001-01-04,0.255387,0.276903,0.251645,0.271524,739396000
3,2001-01-05,0.245097,0.260064,0.240419,0.253515,412356000
4,2001-01-08,0.247903,0.254217,0.238548,0.253515,373699200


In [ ]:
artifacts_root: artifacts


data_ingestion:
  root_dir: artifacts/data_ingestion
  source_URL: https://drive.google.com/file/d/1vlhZ5c7abUKF8xXERIw6m9Te8fW7ohw3/view?usp=sharing
  local_data_file: artifacts/data_ingestion/data.zip
  unzip_dir: artifacts/data_ingestion



In [ ]:
import pandas as pd
from pathlib import Path
def save_csv(data: pd.DataFrame, file_path: Path) -> None:
    """Saves the DataFrame to a CSV file.

    Args:
        data (pd.DataFrame): The DataFrame to save.
        file_path (Path): The path where the CSV file will be saved.
    """
    try:
        data.to_csv(file_path, index=False)
        logger.info(f"Data saved to {file_path}")
    except Exception as e:
        logger.error(f"Error saving data to {file_path}: {e}")
        raise e
